# Q-Shield Final Audit — All Critical Experiments

Runs all pre-submission audits and generates final tables/figures for the paper.

## What this notebook does (run in order):

1. **URL overlap check** — detect data leakage between Trad and CIC
2. **Per-dataset Grad-CAM** — verify attention patterns are not dataset artifacts
3. **Inference time benchmarks** — concrete numbers for mobile-deployable claim
4. **Threshold calibration** — build FNR vs threshold trade-off (Table VII)
5. **Per-version analysis** — how does AUC vary across QR versions in CIC?
6. **Multi-seed stability** — train 3 seeds, report mean±std (OPTIONAL, ~3h)

**Total time on A100:** ~1.5 hours (all except multi-seed)

**Author:** Nicolas A. Llerena Silva (UTEC)

In [ ]:
# 0. SETUP

import sys, os, glob, time
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

import os
from pathlib import Path
BASE = '/content/drive/MyDrive/Proyecto_Quishing_Detection_Nicolas'
assert os.path.exists(BASE), f'BASE not found: {BASE}'
print(f'BASE = {BASE}')

!pip install -q torch torchvision scikit-learn matplotlib seaborn tqdm shap

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (roc_auc_score, f1_score, precision_score,
                             recall_score, roc_curve, confusion_matrix,
                             brier_score_loss)
from sklearn.model_selection import train_test_split
from PIL import Image
import pickle, zipfile, random, json, gc
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Audit results will accumulate here
audit_results = {}

---
## 1. URL OVERLAP CHECK (data leakage audit)

**Goal:** Verify that Trad and CIC do not share URLs (which would inflate metrics).

If overlap > 1%, we must deduplicate or report this as a caveat.

In [ ]:
# 1. URL OVERLAP CHECK

# CIC provides URLs in CSVs. Trad does NOT include URL strings in the
# pickled dataset (only the binary matrices and labels), per their GitHub.
# But we can look for CSVs in Drive just in case.

cic_benign_csv = None
cic_mal_csv = None

# Find CIC URL csvs
for root, dirs, files in os.walk(BASE):
    for f in files:
        if f.endswith('.csv') and 'url' in f.lower():
            fp = os.path.join(root, f)
            if 'benign' in f.lower() or 'benign' in root.lower():
                cic_benign_csv = fp
            elif 'malicious' in f.lower() or 'malicious' in root.lower():
                cic_mal_csv = fp

print(f'CIC benign URLs CSV: {cic_benign_csv}')
print(f'CIC malicious URLs CSV: {cic_mal_csv}')

# Trad does NOT distribute URLs with the dataset per the paper
# (only the pre-generated QR matrices).
# This means we CANNOT do direct URL overlap check.
# But we CAN assess dataset-level independence:
# - Trad paper: URLs from PhishTank (Mar 2025) + Alexa top-1M
# - CIC paper: URLs from different PhishTank crawl + Majestic top-1M
# Different sources + different time windows = low collision probability

audit_results['url_overlap'] = {
    'trad_provides_urls': False,
    'cic_provides_urls': cic_benign_csv is not None and cic_mal_csv is not None,
    'can_directly_verify': False,
    'indirect_evidence': 'Different source corpora + time windows per each paper',
    'risk_level': 'LOW (but unverified)'
}

print('\nConclusion:')
print('- Trad dataset does NOT distribute URL strings (only binary QR matrices)')
print('- Direct URL overlap check is NOT possible')
print('- Indirect evidence (different source papers, different time windows)')
print('  suggests low overlap probability')
print('- Will report as acknowledged limitation in paper')

# If CIC CSVs exist, show sample URLs
if cic_benign_csv:
    df_b = pd.read_csv(cic_benign_csv).head(3)
    print(f'\nCIC benign sample:')
    print(df_b)

---
## 2. LOAD MODEL + DATA (shared by remaining sections)

In [ ]:
# 2.1 MODEL DEFINITION (matches v3 in notebook 06)

class MobileNetV2Embedding(nn.Module):
    def __init__(self, emb_dim=128, pretrained=False, dropout=0.35):
        super().__init__()
        mn = models.mobilenet_v2(weights=None)
        self.features = mn.features
        self.features[0][0] = nn.Conv2d(1, 32, 3, stride=2, padding=1, bias=False)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.projection = nn.Sequential(
            nn.Linear(1280, 512), nn.BatchNorm1d(512), nn.ReLU(True),
            nn.Dropout(dropout), nn.Linear(512, emb_dim),
        )
    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        return F.normalize(self.projection(x), p=2, dim=1)

class QRClassifier(nn.Module):
    def __init__(self, backbone, emb_dim=128):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Sequential(
            nn.Linear(emb_dim, 512), nn.BatchNorm1d(512), nn.ReLU(True), nn.Dropout(0.4),
            nn.Linear(512, 128), nn.BatchNorm1d(128), nn.ReLU(True), nn.Dropout(0.3),
            nn.Linear(128, 32), nn.ReLU(True), nn.Dropout(0.2),
            nn.Linear(32, 1),
        )
    def forward(self, x):
        return self.head(self.backbone(x))

# Load checkpoint
ckpt_candidates = [
    os.path.join(BASE, 'classifier_v3_phase2.pth'),
    os.path.join(BASE, 'classifier_phase2.pth'),
]
CKPT = next((p for p in ckpt_candidates if os.path.exists(p)), None)
assert CKPT is not None, 'No classifier checkpoint found'
print(f'Loading: {CKPT}')

classifier = QRClassifier(MobileNetV2Embedding(128, dropout=0.35), emb_dim=128).to(device)
classifier.load_state_dict(torch.load(CKPT, map_location=device))
classifier.eval()
print(f'Model loaded: {sum(p.numel() for p in classifier.parameters()):,} params')

In [ ]:
# 2.2 LOAD DATASETS

WORK = '/content/qshield_data'
os.makedirs(WORK, exist_ok=True)

# Trad
trad_dir = os.path.join(WORK, 'trad')
if not os.path.exists(os.path.join(trad_dir, 'qr_codes_29.pickle')):
    os.makedirs(trad_dir, exist_ok=True)
    with zipfile.ZipFile(os.path.join(BASE, 'QuishingDataset.zip')) as z:
        z.extractall(trad_dir)
with open(os.path.join(trad_dir, 'qr_codes_29.pickle'), 'rb') as f:
    trad_qr = pickle.load(f)
with open(os.path.join(trad_dir, 'qr_codes_29_labels.pickle'), 'rb') as f:
    trad_labels = pickle.load(f)

# CIC
cic_b_dir = os.path.join(WORK, 'cic_benign')
cic_m_dir = os.path.join(WORK, 'cic_malicious')
if not os.path.exists(cic_b_dir) or len(os.listdir(cic_b_dir)) == 0:
    os.makedirs(cic_b_dir, exist_ok=True)
    with zipfile.ZipFile(os.path.join(BASE, 'QR_benign_430K.zip')) as z:
        z.extractall(cic_b_dir)
if not os.path.exists(cic_m_dir) or len(os.listdir(cic_m_dir)) == 0:
    os.makedirs(cic_m_dir, exist_ok=True)
    with zipfile.ZipFile(os.path.join(BASE, 'QR_malicious_576K.zip')) as z:
        z.extractall(cic_m_dir)

cic_b_files = sorted(glob.glob(os.path.join(cic_b_dir, '**', '*.png'), recursive=True))
cic_m_files = sorted(glob.glob(os.path.join(cic_m_dir, '**', '*.png'), recursive=True))
random.seed(SEED)
cic_b_files = random.sample(cic_b_files, min(10000, len(cic_b_files)))
cic_m_files = random.sample(cic_m_files, min(10000, len(cic_m_files)))

# Splits (same as notebook 06/08 with SEED=42)
idx_tr, idx_val = train_test_split(np.arange(len(trad_labels)), test_size=0.2,
                                    stratify=trad_labels, random_state=SEED)
qr_val = trad_qr[idx_val]
lab_val = trad_labels[idx_val]

sb = int(len(cic_b_files)*0.8); sm = int(len(cic_m_files)*0.8)
cic_b_val_files = cic_b_files[sb:]
cic_m_val_files = cic_m_files[sm:]

print(f'Trad val: {len(qr_val)}')
print(f'CIC val: {len(cic_b_val_files)+len(cic_m_val_files)}')

In [ ]:
# 2.3 HELPER FUNCTIONS

def prepare_trad_tensor(arr):
    t = torch.from_numpy(arr.astype(np.float32)).unsqueeze(0).unsqueeze(0)
    t = F.interpolate(t, size=(224, 224), mode='bilinear', align_corners=False)
    return t.to(device)

def prepare_cic_tensor(path):
    img = Image.open(path).convert('L').resize((224, 224))
    t = torch.from_numpy(np.array(img, dtype=np.float32)/255.0).unsqueeze(0).unsqueeze(0)
    return t.to(device)

def get_predictions_trad():
    probs = []
    classifier.eval()
    with torch.no_grad():
        for i in range(0, len(qr_val), 64):
            batch = qr_val[i:i+64]
            tensors = torch.cat([prepare_trad_tensor(q) for q in batch], dim=0)
            p = torch.sigmoid(classifier(tensors)).cpu().numpy().flatten()
            probs.extend(p)
    return np.array(probs)

def get_predictions_cic():
    probs, labels = [], []
    classifier.eval()
    with torch.no_grad():
        for cls, files in [(0, cic_b_val_files), (1, cic_m_val_files)]:
            for i in range(0, len(files), 64):
                batch = files[i:i+64]
                tensors = torch.cat([prepare_cic_tensor(p) for p in batch], dim=0)
                p = torch.sigmoid(classifier(tensors)).cpu().numpy().flatten()
                probs.extend(p)
                labels.extend([cls]*len(batch))
    return np.array(probs), np.array(labels)

print('Helpers ready.')

---
## 3. PER-DATASET GRAD-CAM AUDIT

**Goal:** verify that the left-right attention pattern is dataset-independent,
i.e., appears inside BOTH Trad and CIC separately.

In [ ]:
# 3. GRAD-CAM PER DATASET

class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.gradients = None
        self.activations = None
        target_layer.register_forward_hook(lambda m, i, o: setattr(self, 'activations', o.detach()))
        target_layer.register_full_backward_hook(lambda m, gi, go: setattr(self, 'gradients', go[0].detach()))
    def __call__(self, x):
        self.model.eval()
        logit = self.model(x)
        self.model.zero_grad()
        logit.backward(torch.ones_like(logit))
        weights = self.gradients.mean(dim=(2,3), keepdim=True)
        cam = F.relu((weights * self.activations).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=(224, 224), mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        if cam.max() > 0:
            cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam

gradcam = GradCAM(classifier, classifier.backbone.features[-1])
N_PER_GROUP = 150

# --- Trad per-class averages ---
print('Computing Grad-CAM on Trad validation set...')
trad_b_idx = np.where(lab_val == 0)[0][:N_PER_GROUP]
trad_p_idx = np.where(lab_val == 1)[0][:N_PER_GROUP]

trad_b_cams, trad_p_cams = [], []
for i in tqdm(trad_b_idx, desc='Trad benign'):
    t = prepare_trad_tensor(qr_val[i]); t.requires_grad_(True)
    with torch.enable_grad():
        trad_b_cams.append(gradcam(t))
for i in tqdm(trad_p_idx, desc='Trad phish'):
    t = prepare_trad_tensor(qr_val[i]); t.requires_grad_(True)
    with torch.enable_grad():
        trad_p_cams.append(gradcam(t))

# --- CIC per-class averages ---
print('Computing Grad-CAM on CIC validation set...')
cic_b_cams, cic_p_cams = [], []
for p in tqdm(cic_b_val_files[:N_PER_GROUP], desc='CIC benign'):
    t = prepare_cic_tensor(p); t.requires_grad_(True)
    with torch.enable_grad():
        cic_b_cams.append(gradcam(t))
for p in tqdm(cic_m_val_files[:N_PER_GROUP], desc='CIC phish'):
    t = prepare_cic_tensor(p); t.requires_grad_(True)
    with torch.enable_grad():
        cic_p_cams.append(gradcam(t))

avg_trad_b = np.mean(trad_b_cams, axis=0)
avg_trad_p = np.mean(trad_p_cams, axis=0)
avg_cic_b = np.mean(cic_b_cams, axis=0)
avg_cic_p = np.mean(cic_p_cams, axis=0)

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
for col, (data, title) in enumerate([
    (avg_trad_b, 'Trad — Benign'),
    (avg_trad_p, 'Trad — Phishing'),
    (avg_trad_p - avg_trad_b, 'Trad — Difference'),
]):
    cmap = 'RdBu_r' if 'Difference' in title else 'jet'
    vmax = abs(data).max() if 'Difference' in title else 1.0
    vmin = -vmax if 'Difference' in title else 0
    im = axes[0, col].imshow(data, cmap=cmap, vmin=vmin, vmax=vmax)
    axes[0, col].set_title(title, fontweight='bold'); axes[0, col].axis('off')
    plt.colorbar(im, ax=axes[0, col], shrink=0.7)

for col, (data, title) in enumerate([
    (avg_cic_b, 'CIC — Benign'),
    (avg_cic_p, 'CIC — Phishing'),
    (avg_cic_p - avg_cic_b, 'CIC — Difference'),
]):
    cmap = 'RdBu_r' if 'Difference' in title else 'jet'
    vmax = abs(data).max() if 'Difference' in title else 1.0
    vmin = -vmax if 'Difference' in title else 0
    im = axes[1, col].imshow(data, cmap=cmap, vmin=vmin, vmax=vmax)
    axes[1, col].set_title(title, fontweight='bold'); axes[1, col].axis('off')
    plt.colorbar(im, ax=axes[1, col], shrink=0.7)

fig.suptitle('Per-Dataset Grad-CAM Validation', fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'fig_gradcam_per_dataset.png'), dpi=300, bbox_inches='tight')
plt.savefig('/content/fig_gradcam_per_dataset.png', dpi=300, bbox_inches='tight')
plt.show()

# Correlation between difference maps across datasets
diff_trad_flat = (avg_trad_p - avg_trad_b).flatten()
diff_cic_flat = (avg_cic_p - avg_cic_b).flatten()
corr = np.corrcoef(diff_trad_flat, diff_cic_flat)[0, 1]
print(f'\nSpatial correlation between Trad and CIC difference maps: {corr:.4f}')

audit_results['gradcam_per_dataset_correlation'] = float(corr)
if corr > 0.5:
    print('Interpretation: HIGH correlation — attention pattern is consistent across datasets (valid signal)')
elif corr > 0.2:
    print('Interpretation: MODERATE correlation — some shared pattern, some dataset-specific')
else:
    print('Interpretation: LOW correlation — attention may be dataset artifact (red flag)')

---
## 4. INFERENCE TIME BENCHMARKS

In [ ]:
# 4. INFERENCE BENCHMARKS

classifier.eval()
N_WARMUP = 20
N_MEASURE = 100

# Warmup
dummy = torch.randn(1, 1, 224, 224).to(device)
with torch.no_grad():
    for _ in range(N_WARMUP):
        classifier(dummy)

# Single-image latency (GPU)
if device.type == 'cuda':
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        for _ in range(N_MEASURE):
            classifier(dummy)
    torch.cuda.synchronize()
    gpu_latency_ms = (time.perf_counter() - t0) / N_MEASURE * 1000
else:
    gpu_latency_ms = None

# CPU latency
classifier_cpu = QRClassifier(MobileNetV2Embedding(128, dropout=0.35), emb_dim=128)
classifier_cpu.load_state_dict(torch.load(CKPT, map_location='cpu'))
classifier_cpu.eval()
dummy_cpu = torch.randn(1, 1, 224, 224)
with torch.no_grad():
    for _ in range(N_WARMUP):
        classifier_cpu(dummy_cpu)
t0 = time.perf_counter()
with torch.no_grad():
    for _ in range(N_MEASURE):
        classifier_cpu(dummy_cpu)
cpu_latency_ms = (time.perf_counter() - t0) / N_MEASURE * 1000

# Model size on disk
model_size_mb = os.path.getsize(CKPT) / 1024 / 1024

# Batch throughput
dummy_batch = torch.randn(32, 1, 224, 224).to(device)
if device.type == 'cuda':
    with torch.no_grad():
        for _ in range(5):
            classifier(dummy_batch)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        for _ in range(50):
            classifier(dummy_batch)
    torch.cuda.synchronize()
    throughput = (50 * 32) / (time.perf_counter() - t0)
else:
    throughput = None

print('='*60)
print(' INFERENCE BENCHMARKS')
print('='*60)
print(f'Model size on disk:         {model_size_mb:.1f} MB')
print(f'Parameters:                 {sum(p.numel() for p in classifier.parameters()):,}')
print(f'CPU single-image latency:   {cpu_latency_ms:.2f} ms')
if gpu_latency_ms:
    print(f'GPU ({torch.cuda.get_device_name(0)}) latency: {gpu_latency_ms:.2f} ms')
if throughput:
    print(f'GPU throughput:             {throughput:.0f} images/sec')

audit_results['inference'] = {
    'model_size_mb': round(model_size_mb, 2),
    'params': sum(p.numel() for p in classifier.parameters()),
    'cpu_latency_ms': round(cpu_latency_ms, 2),
    'gpu_latency_ms': round(gpu_latency_ms, 2) if gpu_latency_ms else None,
    'gpu_throughput_imgs_per_sec': round(throughput, 0) if throughput else None,
    'gpu_name': torch.cuda.get_device_name(0) if device.type == 'cuda' else None,
}

---
## 5. THRESHOLD CALIBRATION — critical for security deployment

In [ ]:
# 5. THRESHOLD CALIBRATION ANALYSIS

# Get combined val predictions
print('Computing predictions on Trad val...')
probs_trad = get_predictions_trad()
true_trad = lab_val

print('Computing predictions on CIC val...')
probs_cic, true_cic = get_predictions_cic()

probs_all = np.concatenate([probs_trad, probs_cic])
true_all = np.concatenate([true_trad, true_cic])

print(f'Total val samples: {len(probs_all)}')
print(f'AUC (threshold-independent): {roc_auc_score(true_all, probs_all):.4f}')

In [ ]:
# 5.1 THRESHOLD SWEEP — FNR vs Precision Trade-off

thresholds = np.arange(0.05, 0.95, 0.025)
results = []
for t in thresholds:
    preds = (probs_all >= t).astype(int)
    tp = ((preds == 1) & (true_all == 1)).sum()
    tn = ((preds == 0) & (true_all == 0)).sum()
    fp = ((preds == 1) & (true_all == 0)).sum()
    fn = ((preds == 0) & (true_all == 1)).sum()
    prec = tp / (tp + fp + 1e-8)
    rec = tp / (tp + fn + 1e-8)
    fnr = fn / (fn + tp + 1e-8)
    fpr = fp / (fp + tn + 1e-8)
    f1 = 2*prec*rec / (prec+rec+1e-8)
    results.append({'threshold': t, 'precision': prec, 'recall': rec, 'fnr': fnr, 'fpr': fpr, 'f1': f1})
df_t = pd.DataFrame(results)

# Find threshold that hits FNR <= 0.10
targets = {'fnr_0.10': 0.10, 'fnr_0.05': 0.05, 'f1_max': None}
found = {}
valid = df_t[df_t['fnr'] <= 0.10]
if len(valid):
    row = valid.iloc[-1]  # highest threshold still meeting FNR constraint
    found['fnr_0.10'] = row.to_dict()
valid = df_t[df_t['fnr'] <= 0.05]
if len(valid):
    row = valid.iloc[-1]
    found['fnr_0.05'] = row.to_dict()
f1_row = df_t.loc[df_t['f1'].idxmax()]
found['f1_max'] = f1_row.to_dict()
default_row = df_t[np.isclose(df_t['threshold'], 0.5)].iloc[0]
found['default_0.5'] = default_row.to_dict()

print('='*80)
print(' PAPER TABLE VII — Threshold Calibration')
print('='*80)
print(f"{'Operating Point':<20} {'Threshold':>10} {'Precision':>10} {'Recall':>8} {'FNR':>8} {'F1':>8}")
print('-'*80)
for name, row in [('Default (0.5)', found['default_0.5']),
                   ('Maximize F1', found['f1_max']),
                   ('FNR <= 0.10', found.get('fnr_0.10')),
                   ('FNR <= 0.05', found.get('fnr_0.05'))]:
    if row is None: continue
    print(f"{name:<20} {row['threshold']:>10.3f} {row['precision']:>10.4f} {row['recall']:>8.4f} {row['fnr']:>8.4f} {row['f1']:>8.4f}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(df_t['threshold'], df_t['fnr'], lw=2, color='#e74c3c', label='FNR')
axes[0].plot(df_t['threshold'], df_t['fpr'], lw=2, color='#3498db', label='FPR')
axes[0].axhline(0.10, color='gray', ls='--', alpha=0.5, label='FNR=0.10 target')
axes[0].axvline(0.5, color='black', ls=':', alpha=0.5, label='Default 0.5')
axes[0].set_xlabel('Decision threshold')
axes[0].set_ylabel('Error rate')
axes[0].set_title('Error Rates vs Threshold', fontweight='bold')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(df_t['threshold'], df_t['precision'], lw=2, color='#2ecc71', label='Precision')
axes[1].plot(df_t['threshold'], df_t['recall'], lw=2, color='#9b59b6', label='Recall')
axes[1].plot(df_t['threshold'], df_t['f1'], lw=2, color='#f39c12', label='F1')
axes[1].axvline(0.5, color='black', ls=':', alpha=0.5)
axes[1].set_xlabel('Decision threshold')
axes[1].set_ylabel('Metric value')
axes[1].set_title('Precision/Recall/F1 vs Threshold', fontweight='bold')
axes[1].legend(); axes[1].grid(alpha=0.3)

fig.suptitle('Q-Shield Operating Point Calibration', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'fig_threshold_calibration.png'), dpi=300, bbox_inches='tight')
plt.savefig('/content/fig_threshold_calibration.png', dpi=300, bbox_inches='tight')
plt.show()

# Brier score + ECE
brier = brier_score_loss(true_all, probs_all)
# ECE: 10 bins
n_bins = 10
bin_edges = np.linspace(0, 1, n_bins+1)
ece = 0
for i in range(n_bins):
    mask = (probs_all >= bin_edges[i]) & (probs_all < bin_edges[i+1])
    if mask.sum() > 0:
        bin_acc = true_all[mask].mean()
        bin_conf = probs_all[mask].mean()
        ece += (mask.sum() / len(probs_all)) * abs(bin_acc - bin_conf)
print(f'\nBrier score:           {brier:.4f} (lower is better)')
print(f'Expected Calibration Error (ECE): {ece:.4f}')

audit_results['threshold_calibration'] = {
    'default_0.5': found['default_0.5'],
    'f1_max': found['f1_max'],
    'fnr_0.10': found.get('fnr_0.10'),
    'fnr_0.05': found.get('fnr_0.05'),
    'brier_score': brier,
    'ece': ece,
}
df_t.to_csv(os.path.join(BASE, 'threshold_calibration.csv'), index=False)

---
## 6. PER-VERSION ANALYSIS ON CIC (if metadata available)

In [ ]:
# 6. PER-VERSION ANALYSIS

# CIC PNGs have varying dimensions which correlate with QR version.
# We can approximate per-version AUC by bucketing by image size.

print('Estimating QR version from PNG dimensions...')
sizes_b = []
sizes_m = []
for p in tqdm(cic_b_val_files, desc='Sizing benign'):
    sizes_b.append(Image.open(p).size[0])
for p in tqdm(cic_m_val_files, desc='Sizing mal'):
    sizes_m.append(Image.open(p).size[0])

sizes_all = np.array(sizes_b + sizes_m)
print(f'Size range: {sizes_all.min()} - {sizes_all.max()}')
print(f'Unique sizes: {len(np.unique(sizes_all))}')
print(f'Size histogram:')
vals, cnts = np.unique(sizes_all, return_counts=True)
for v, c in zip(vals, cnts):
    print(f'  {v}px: {c:,} samples')

# Group by size buckets (approximate QR versions)
# QR versions produce: 21, 25, 29, ..., 177 modules, each px-scaled
probs_cic_val = probs_cic
true_cic_val = true_cic

# Size per sample (in order benign then malicious)
sizes_per_sample = np.array(sizes_b + sizes_m)

# Bucket by size quartiles
q25, q50, q75 = np.quantile(sizes_per_sample, [0.25, 0.5, 0.75])
buckets = [
    ('Small', lambda s: s <= q25),
    ('Medium-Small', lambda s: (s > q25) & (s <= q50)),
    ('Medium-Large', lambda s: (s > q50) & (s <= q75)),
    ('Large', lambda s: s > q75),
]

per_size_results = []
for name, fn in buckets:
    mask = fn(sizes_per_sample)
    if mask.sum() < 10: continue
    b_p = probs_cic_val[mask]
    b_t = true_cic_val[mask]
    if len(np.unique(b_t)) < 2: continue
    auc = roc_auc_score(b_t, b_p)
    preds = (b_p >= 0.5).astype(int)
    f1 = f1_score(b_t, preds) if len(np.unique(preds)) > 1 else 0
    fnr = 1 - recall_score(b_t, preds) if 1 in b_t else 0
    per_size_results.append({
        'bucket': name, 'n': int(mask.sum()),
        'size_min': int(sizes_per_sample[mask].min()),
        'size_max': int(sizes_per_sample[mask].max()),
        'AUC': round(auc, 4), 'F1': round(f1, 4), 'FNR': round(fnr, 4),
    })

print('\n' + '='*80)
print(' PER-SIZE AUC ON CIC (approximates per-version analysis)')
print('='*80)
df_size = pd.DataFrame(per_size_results)
print(df_size.to_string(index=False))
df_size.to_csv(os.path.join(BASE, 'per_size_results.csv'), index=False)
audit_results['per_size_analysis'] = per_size_results

---
## 7. FINAL AUDIT REPORT

In [ ]:
# 7. SAVE AUDIT REPORT

# Make JSON-serializable
def to_serializable(obj):
    if isinstance(obj, dict):
        return {k: to_serializable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [to_serializable(x) for x in obj]
    if isinstance(obj, (np.integer,)): return int(obj)
    if isinstance(obj, (np.floating,)): return float(obj)
    if isinstance(obj, np.ndarray): return obj.tolist()
    return obj

serializable = to_serializable(audit_results)
report_path = os.path.join(BASE, 'final_audit_report.json')
with open(report_path, 'w') as f:
    json.dump(serializable, f, indent=2)

print('='*70)
print(' FINAL AUDIT COMPLETE')
print('='*70)
print(json.dumps(serializable, indent=2))
print(f'\nAll saved to {report_path}')